In [ ]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import ast
import copy
import warnings
import copy
from pprint import pprint

# Suppress pandas SettingWithCopyWarning
warnings.filterwarnings('ignore', category=pd.errors.SettingWithCopyWarning)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print("Libraries loaded.")

Libraries loaded.


# Read in resulting data

In [2]:

# Load Ground Truth
gt_data = []
with open('data/fever-data/dev.jsonl', 'r') as f: # Adjust path to your GT file
    for line in f:
        gt_data.append(json.loads(line))=
df_gt = pd.DataFrame(gt_data)


In [3]:

df_gt


,id,verifiable,label,claim,evidence
0,91198,NOT VERIFIABLE,NOT ENOUGH INFO,Colin Kaepernick became a starting quarterback...,"[[[108548, None, None, None]]]"
1,194462,NOT VERIFIABLE,NOT ENOUGH INFO,Tilda Swinton is a vegan.,"[[[227768, None, None, None]]]"
2,137334,VERIFIABLE,SUPPORTS,Fox 2000 Pictures released the film Soul Food.,"[[[289914, 283015, Soul_Food_-LRB-film-RRB-, 0..."
3,166626,NOT VERIFIABLE,NOT ENOUGH INFO,Anne Rice was born in New Jersey.,"[[[191656, None, None, None], [191657, None, N..."
4,111897,VERIFIABLE,REFUTES,Telemundo is a English-language television net...,"[[[131371, 146144, Telemundo, 0]], [[131371, 1..."
...,...,...,...,...,...
19993,8538,VERIFIABLE,REFUTES,Hermit crabs are arachnids.,"[[[15450, 19262, Hermit_crab, 0], [15450, 1926..."
19994,145641,VERIFIABLE,REFUTES,Michael Hutchence died on a boat.,"[[[168967, 182663, Michael_Hutchence, 15]]]"
19995,87517,VERIFIABLE,SUPPORTS,The Cyclades are located to the southeast of G...,"[[[104709, 118125, Cyclades, 0]]]"
19996,111816,NOT VERIFIABLE,NOT ENOUGH INFO,Theresa May worked the docks.,"[[[131223, None, None, None]]]"


In [4]:
df_gt['label'].value_counts()

label
NOT ENOUGH INFO    6666
SUPPORTS           6666
REFUTES            6666
Name: count, dtype: int64

In [5]:

def get_retrived_docs(file):
    with open(file, 'r') as f:
        retrieved_docs = json.load(f)
    retrieved_df = pd.DataFrame(retrieved_docs)
    return dict(zip(retrieved_df['claim'], retrieved_df['retrieved_documents']))


In [ ]:

def read_all(folder, close_book_fp):
    df_closed = pd.read_csv(close_book_fp)
    df_closed = df_closed.rename(columns={'pred': 'pred_closed'})    
    
    # Note: all generated files are setup to be following the same order
    df_open_bm25 = pd.read_csv(f'{folder}/dev_llm_classification_results_merged_bm25_top5.csv')
    df_open_bm25 = df_open_bm25[['claim', 'classification']].rename(columns={'classification': 'pred_open_bm25'})
    df_open_bm25_retrieved_dict = get_retrived_docs('../results/dev_claim_retrieved_docs_bm25_top5.json')
    df_open_bm25['retrieved_documents_bm25'] = df_open_bm25['claim'].map(df_open_bm25_retrieved_dict)
    df_open_bm25 = df_open_bm25.rename(columns={'pred_open_bm25': 'classification'})

    df_open_dense_mini = pd.read_csv(f'{folder}/dev_llm_classification_results_merged_dense_miniLM_top5.csv')
    df_open_dense_mini_retrieved_dict = get_retrived_docs('../results/dev_claim_retrieved_docs_dense_miniLM_top5.json')
    df_open_dense_mini['retrieved_documents_dense_mini'] = df_open_dense_mini['claim'].map(df_open_dense_mini_retrieved_dict)

    df_open_dense_qwen3 = pd.read_csv(f'{folder}/dev_llm_classification_results_merged_dense_qwen3_top5.csv')
    df_open_dense_qwen3_retrieved_dict = get_retrived_docs('../results/dev_claim_retrieved_docs_dense_qwen3_top5.json')
    df_open_dense_qwen3['retrieved_documents_dense_qwen3'] = df_open_dense_qwen3['claim'].map(df_open_dense_qwen3_retrieved_dict)
    
    # Ensure row-wise alignment by claim
    assert (df_gt['claim'].values == df_closed['claim'].values).all()
    assert (df_gt['claim'].values == df_open_bm25['claim'].values).all()
    assert (df_gt['claim'].values == df_open_dense_mini['claim'].values).all()
    assert (df_gt['claim'].values == df_open_dense_qwen3['claim'].values).all()
    assert df_closed.shape[0] == df_open_bm25.shape[0] == df_open_dense_mini.shape[0] == df_open_dense_qwen3.shape[0]
    
    df_merged = copy.deepcopy(df_gt)
    df_merged['classification_closed'] = df_closed['pred_closed']
    df_merged['classification_bm25'] = df_open_bm25['classification']
    df_merged['classification_mini'] = df_open_dense_mini['classification']
    df_merged['classification_qwen3'] = df_open_dense_qwen3['classification']

    df_merged['retrieved_documents_bm25'] = df_open_bm25['retrieved_documents_bm25']
    df_merged['retrieved_documents_dense_mini'] = df_open_dense_mini['retrieved_documents_dense_mini']
    df_merged['retrieved_documents_dense_qwen3'] = df_open_dense_qwen3['retrieved_documents_dense_qwen3']
    assert df_merged.shape[0] == df_merged.shape[0]

    return df_merged



In [ ]:
folder = '../results/dev_res_qwen25'
df_qwen25_merged = read_all(folder, '../results/dev_res_qwen25/fever_closedbook_qwen2.5_7b_instruct_dev_subset.csv')

In [ ]:

folder = '../results/dev_res_llama3_newprompt'
df_llama3_newprompt_merged = read_all(folder, '../results/dev_res_llama3_newprompt/fever_closedbook_llama3.1_8b_instruct_dev_subset.csv')

In [11]:
def compute_label_acc(df_merged):
    res ={}
    for col in ['closed','bm25', 'mini', 'qwen3']:
        res[col] = len(df_merged[df_merged['label']==df_merged[f'classification_{col}']])/len(df_merged)
        # print(f'{col}:', len(df_merged[df_merged['label']==df_merged[f'classification_{col}']])/len(df_merged), end=', ')
    return res

In [ ]:
results = {
    'Qwen2.5': compute_label_acc(df_qwen25_merged),
    'Llama3(newprompt)': compute_label_acc(df_llama3_newprompt_merged)
}

df_results = pd.DataFrame(results)
print(df_results)

         Qwen2.5  Llama3(newprompt)
closed  0.262676           0.377888
bm25    0.615162           0.576408
mini    0.596410           0.557006
qwen3   0.664116           0.614311


In [ ]:
# retrieval recall analysis:Is miniLM retrieval retrieved unrelevant documents the most compare to BM25?

def colleact_retrieved_docs_vs_classification_results(col='retrieved_documents_dense_mini'):
    df = copy.deepcopy(df_gt[['claim', 'evidence', 'label']])
    
    df['retrieved_documents'] = df_llama3_newprompt_merged[col] # any would be the same
    return df

def tidy_docid(df, top_k=5):
    df['evidence_doc_id'] = df['evidence'].apply(
        lambda docs: list(dict.fromkeys([x[0][-2] for x in docs])) # Keep the order
    )

    df['retrieved_doc_id'] = df['retrieved_documents'].apply(
        lambda docs: list(dict.fromkeys([x['doc_id'] for x in docs[:top_k]]))  # Keep the order
    )
    return df

def calculate_metrics(row):
    # Convert lists to sets for fast comparison
    ground_truth = set(row['evidence_doc_id'])
    retrieved = set(row['retrieved_doc_id'])
    
    # Calculate Intersection (True Positives)
    # The set of items that appear in BOTH lists
    true_positives = ground_truth.intersection(retrieved)
    len_tp = len(true_positives)
    
    # Precision: TP / Total Retrieved
    if len(retrieved) > 0:
        precision = len_tp / len(retrieved)
    else:
        precision = 0.0
        
    # Recall: TP / Total Expected (Evidence)
    if len(ground_truth) > 0:
        recall = len_tp / len(ground_truth)
    else:
        recall = 0.0
        
    # F1 Score: Harmonic mean of Precision and Recall
    if (precision + recall) > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0.0

    return pd.Series([precision, recall, f1], index=['precision', 'recall', 'f1'])



In [14]:
def gettopk_retrieved_docs(df, top_k=5):
    df = tidy_docid(df, top_k)

    df = df[df['label']!='NOT ENOUGH INFO']

    df[['precision', 'recall', 'f1']] = df.apply(calculate_metrics, axis=1)
    return df[['precision', 'recall', 'f1']].mean().to_dict()

In [15]:

def get_relevance_scores(df, top_k=5):
    df = tidy_docid(df, top_k=top_k)
    df_temp = df[df['label']!='NOT ENOUGH INFO']
    relevance_scores = df_temp.apply(
        lambda row: [1 if x in row['evidence_doc_id'] else 0 for x in row['retrieved_doc_id'] ],
        axis=1).to_list()
    return relevance_scores


In [16]:
import numpy as np

def dcg_at_k(relevance_scores, k):
    k = min(len(relevance_scores), k)
    discounts = np.log2(np.arange(2, k + 2))
    gains = np.array(relevance_scores[:k])
    return np.sum(gains / discounts)

def ndcg_at_k(relevance_scores, k):
    actual_dcg = dcg_at_k(relevance_scores, k)
    ideal_scores = sorted(relevance_scores, reverse=True)
    ideal_dcg = dcg_at_k(ideal_scores, k)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg

def precision_at_k(relevance_scores, k):
    """Calculates the precision at k."""
    k = min(len(relevance_scores), k)
    # The number of relevant items is simply the sum of the scores (1s)
    num_relevant_items = sum(relevance_scores[:k])
    return num_relevant_items / k

def get_ndcg_precision_scores(relevance_scores, k=5):
    # --- Run the calculations ---
    ndcg_scores = []
    for rel_scores in relevance_scores:
        ndcg_score = ndcg_at_k(rel_scores, k)
        precision_score = precision_at_k(rel_scores, k)
        ndcg_scores.append(ndcg_score)

    # print(f"nDCG@{k}: {np.mean(ndcg_scores):.4f}")
    return np.mean(ndcg_scores)


In [17]:
df_bm25 = colleact_retrieved_docs_vs_classification_results('retrieved_documents_bm25')
df_mini = colleact_retrieved_docs_vs_classification_results('retrieved_documents_dense_mini')
df_qwen3 = colleact_retrieved_docs_vs_classification_results('retrieved_documents_dense_qwen3')

for k in [1, 3, 5]:
    print(f'Top {k} results:\n==============')
    
    bm25_scores = get_relevance_scores(df_bm25, top_k=k)
    nDCG = get_ndcg_precision_scores(bm25_scores, k=k)
    print('df_bm25', 
          gettopk_retrieved_docs(df_bm25, top_k=k), f'nDCG={nDCG:.4f}'
          )
    
    mini_scores = get_relevance_scores(df_mini, top_k=k)
    nDCG = get_ndcg_precision_scores(mini_scores, k=k)
    print('df_mini',
          gettopk_retrieved_docs(df_mini, top_k=k), f'nDCG={nDCG:.4f}'
          )
    
    qwen3_scores = get_relevance_scores(df_qwen3, top_k=k)
    nDCG = get_ndcg_precision_scores(qwen3_scores, k=k)
    print('df_qwen3',   
          gettopk_retrieved_docs(df_qwen3, top_k=k), f'nDCG={nDCG:.4f}'
          )
    
    print('==============\n')
#     break

Top 1 results:
df_bm25 {'precision': 0.2668766876687669, 'recall': 0.2668766876687669, 'f1': 0.2668766876687669} nDCG=0.2669
df_mini {'precision': 0.28997899789979, 'recall': 0.28997899789979, 'f1': 0.28997899789979} nDCG=0.2900
df_qwen3 {'precision': 0.6406390639063907, 'recall': 0.6406390639063907, 'f1': 0.6406390639063907} nDCG=0.6406

Top 3 results:
df_bm25 {'precision': 0.14216421642164218, 'recall': 0.42649264926492647, 'f1': 0.21324632463246324} nDCG=0.3595
df_mini {'precision': 0.15766576657665768, 'recall': 0.472997299729973, 'f1': 0.2364986498649865} nDCG=0.3964
df_qwen3 {'precision': 0.282028202820282, 'recall': 0.8460846084608461, 'f1': 0.42304230423042305} nDCG=0.7630

Top 5 results:
df_bm25 {'precision': 0.10099009900990098, 'recall': 0.504950495049505, 'f1': 0.16831683168316838} nDCG=0.3918
df_mini {'precision': 0.11006600660066006, 'recall': 0.5503300330033003, 'f1': 0.18344334433443346} nDCG=0.4282
df_qwen3 {'precision': 0.17823282328232823, 'recall': 0.891164116411641

In [ ]:
# # TO ANSWER: WHY BM25 retrieved less relevant documents but has more correct classification results with downstream classification task
# CASE STUDY 1: 5.4 Qualitative Analysis: The "Distractor" Problem

# Use qwen25 classification results as example since it is the strongest performance, investigating
# miniLM retrieved unrelevant documents issue
sample = df_qwen25_merged[(
    (df_qwen25_merged['label']=='SUPPORTS') &
    (df_qwen25_merged['classification_closed']=='NOT_ENOUGH_INFO') &
    (df_qwen25_merged['classification_qwen3']=='SUPPORTS') &
    (df_qwen25_merged['classification_mini']=='REFUTES')
)].iloc[0][['claim', 'classification_closed', 'evidence', 'label',
            'classification_mini', 'classification_qwen3', 'classification_bm25',
            'retrieved_documents_dense_mini',
            'retrieved_documents_dense_qwen3',
            'retrieved_documents_bm25'
            ]]

pprint(sample.to_dict())

{'claim': 'Bret Easton Ellis wrote the screenplay for The Canyons.',
 'classification_bm25': 'SUPPORTS',
 'classification_closed': 'NOT_ENOUGH_INFO',
 'classification_mini': 'REFUTES',
 'classification_qwen3': 'SUPPORTS',
 'evidence': [[[58142, 68354, 'Bret_Easton_Ellis', 20]]],
 'label': 'SUPPORTS',
 'retrieved_documents_bm25': [{'doc_id': 'The_Canyons_-LRB-film-RRB-',
                               'score': 13.719900131225586,
                               'text': 'the canyons is a 2013 american erotic '
                                       'thriller-drama film directed by paul '
                                       'schrader and written by bret easton '
                                       'ellis . the film is set in los angeles '
                                       'and stars lindsay lohan , james deen , '
                                       'nolan funk , amanda brooks , and gus '
                                       'van sant . it received a limited '
              

### Qualitative Analysis: The "Distractor" Problem

**To understand - Why did MiniLM fail where BM25 succeeded?**

* **Case Study:** Claim *"Bret Easton Ellis wrote the screenplay for The Canyons."*

* **Result:** BM25 correctly predicted `SUPPORTS`. MiniLM incorrectly predicted `REFUTES`.

* **Root Cause Analysis:**

    * **MiniLM (Active Distractors):** Retrieved documents for *other* works with nearly identical titles (*The Canyon* by Steve Allrich, *Canyons* by Gary Paulsen). These presented **conflicting authors**, confusing the LLM into finding a contradiction.

    * **BM25 (Passive Noise):** Retrieved documents for *other* works by the same author (*Lunar Park* by Bret Easton Ellis). While irrelevant to the specific film, these did not contradict the author's identity.

* **Conclusion:** For verification, **Passive Noise** (irrelevant but neutral) is safer than **Active Distractors** (semantically similar but factually conflicting).

(This explains why although the BM25 retrieval scores is lower than MiniLM, the LLM classification results is higher)


In [ ]:
# CASE STUDY 2: 6: Discussion: Knowledge Regression vs. Correction

# Use qwen25 classification results as example since it is the strongest performance, investigating
# miniLM retrieved unrelevant documents issue
sample = df_qwen25_merged[(
    # (df_qwen25_merged['label']=='SUPPORTS') &
    (df_qwen25_merged['classification_closed']==df_qwen25_merged['label']) &
    (df_qwen25_merged['classification_mini']!=df_qwen25_merged['label']) &
    (df_qwen25_merged['classification_mini']!='NOT ENOUGH INFO')
)].iloc[0][['claim', 'classification_closed', 'evidence', 'label',
            'classification_mini', 'classification_qwen3', 'classification_bm25',
            'retrieved_documents_dense_mini',
            'retrieved_documents_dense_qwen3',
            'retrieved_documents_bm25'
            ]]


pprint(sample.to_dict())


{'claim': 'A&E is a cable and satellite television network.',
 'classification_bm25': 'SUPPORTS',
 'classification_closed': 'SUPPORTS',
 'classification_mini': 'REFUTES',
 'classification_qwen3': 'SUPPORTS',
 'evidence': [[[263235, 260964, 'A&E_-LRB-TV_channel-RRB-', 0]]],
 'label': 'SUPPORTS',
 'retrieved_documents_bm25': [{'doc_id': 'Arts_and_Entertainment',
                               'score': 15.172900199890137,
                               'text': 'arts and entertainment may refer to :  '
                                       'a&e network , an american cable and '
                                       'satellite television network  a&e '
                                       'television networks , the cable '
                                       "network 's parent company  arts and "
                                       'entertainment -lrb- album -rrb- , a '
                                       'hip hop album by american rappers '
                                    

### Regression Case Study: Geographic Distractors
**How RAG overrides correct internal knowledge.**

* **Claim:** *"A&E is a cable and satellite television network."*
* **Behavior:**
    * **Closed-Book:** `SUPPORTS` (Correct). The model knows A&E is a major network.
    * **MiniLM RAG:** `REFUTES` (Incorrect). The model was confused by the retrieved text.
* **Retrieval Analysis:**
    * **Rank 1:** *A&E (Spain and Portugal)*
    * **Rank 2:** *A&E Australia*
    * **Rank 3:** *A&E (TV Channel)* <- **Gold Doc (US Version)**
* **Insight:** MiniLM favored the "visually similar" titles of international branches over the primary US network. The LLM saw the specific definitions (Spanish/Australian) at the top and hallucinated that the general claim was false.
* **Conclusion:** This is a classic **Knowledge Regression**. The retrieval system introduced **Specificity Distractors** that degraded the model's performance compared to no context at all.